<a href="https://colab.research.google.com/github/Umang-Gisma/M505D/blob/main/LR_KNN_for_Bank_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import pandas            as pd
import numpy             as np
import seaborn           as sns
import plotly.express    as px
import matplotlib.pyplot as plt

from imblearn.over_sampling    import SMOTE
from sklearn.preprocessing     import StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler
from sklearn.model_selection   import train_test_split, GridSearchCV
from sklearn.linear_model      import LogisticRegression
from sklearn.neighbors         import KNeighborsClassifier
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics           import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics           import classification_report, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings("ignore")

In [37]:
APPLY_SMOTE     = False
APPLY_SCALER    = True
SHOW_CONFMATRIX = False

In [38]:
df = pd.read_csv("/content/bank_data.csv")
print("Shape of the dataset:", df.shape)
df.head(3)

Shape of the dataset: (10000, 13)


,id,last_name,credit_score,country,gender,age,years_customer,balance_euros,num_products,has_credit_card,is_active,salary_euros,retained
0,15634602,Hargrave,619,Switzerland,f,42,2,0.00,1,1,1,101348.88,0
1,15647311,Hill,608,Austria,f,41,1,83807.86,1,0,1,112542.58,1
2,15619304,Onio,502,Switzerland,f,42,8,159660.80,3,1,0,113931.57,0


In [39]:
df["churn"] = df["retained"].apply(lambda x: 1 if x==0 else 0)

df.drop(["id", "last_name", "retained"], axis=1, inplace=True)

df = df.join(pd.get_dummies(df['gender']))
df.drop("gender", axis=1, inplace=True)

df = df.join(pd.get_dummies(df['country']))
df.drop("country", axis=1, inplace=True)

In [40]:
y = df['churn'].to_numpy()
y = y.astype(float)

df.drop('churn', inplace=True, axis=1)

X = df.values
X = X.astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [41]:
if APPLY_SCALER:
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
else:
    X_train_scaled = X_train


In [42]:
if APPLY_SMOTE:
    sm = SMOTE()
    X_res, y_res = sm.fit_resample(X_train_scaled, y_train)
else:
    X_res = X_train_scaled
    y_res = y_train

## Logistic Regression

In [43]:
lr_clf = LogisticRegression(random_state=42)

# Training
lr_clf.fit(X_res, y_res)

if APPLY_SCALER:
    X_test_scaled = scaler.transform(X_test)
else:
    X_test_scaled = X_test

# Prediction
y_pred = lr_clf.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

if SHOW_CONFMATRIX:
    ConfusionMatrixDisplay.from_estimator(lr_clf, X_test_scaled, y_test)

              precision    recall  f1-score   support

         0.0       0.83      0.96      0.89      1607
         1.0       0.55      0.20      0.29       393

    accuracy                           0.81      2000
   macro avg       0.69      0.58      0.59      2000
weighted avg       0.78      0.81      0.77      2000



## Hyperparameter Tunning for Logistic Regression

In [44]:
param_grid = {
    # "n_neighbors": [1,3,5,7,11,15,100],
    # "weights": ["uniform", "distance"],
    # # "metric": ["euclidean", "manhattan"]
    # "p":[1,2]
}

lrh = LogisticRegression(random_state=42)

grid_search = GridSearchCV(
    estimator=lrh,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
    )

grid_search.fit(X_res, y_res)

best_lrh = grid_search.best_estimator_

print()
print("Best Param:",grid_search.best_estimator_)
print()
print("Best CV Score:",grid_search.best_score_)

if APPLY_SCALER:
    X_test_scaled = scaler.transform(X_test)
else:
    X_test_scaled = X_test

# Prediction
y_pred = best_lrh.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

if SHOW_CONFMATRIX:
    ConfusionMatrixDisplay.from_estimator(best_lrh, X_test_scaled, y_test)

Fitting 5 folds for each of 1 candidates, totalling 5 fits

Best Param: LogisticRegression(random_state=42)

Best CV Score: 0.319390347967328
              precision    recall  f1-score   support

         0.0       0.83      0.96      0.89      1607
         1.0       0.55      0.20      0.29       393

    accuracy                           0.81      2000
   macro avg       0.69      0.58      0.59      2000
weighted avg       0.78      0.81      0.77      2000



##KNeighborsClassifier


In [45]:
knn_clf = KNeighborsClassifier()

# Training
knn_clf.fit(X_res, y_res)

if APPLY_SCALER:
    X_test_scaled = scaler.transform(X_test)
else:
    X_test_scaled = X_test

# Prediction
y_pred = knn_clf.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

if SHOW_CONFMATRIX:
    ConfusionMatrixDisplay.from_estimator(knn_clf, X_test_scaled, y_test)

              precision    recall  f1-score   support

         0.0       0.86      0.94      0.90      1607
         1.0       0.61      0.37      0.46       393

    accuracy                           0.83      2000
   macro avg       0.73      0.65      0.68      2000
weighted avg       0.81      0.83      0.81      2000



## Hyperparameter Tunning for KNN

In [46]:
param_grid = {
    "n_neighbors": [1,3,5,7,11,15,100],
    "weights": ["uniform", "distance"],
    # "metric": ["euclidean", "manhattan"]
    "p":[1,2]
}

knn = KNeighborsClassifier()

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
    )

grid_search.fit(X_res, y_res)

best_knn = grid_search.best_estimator_

print()
print("Best Param:",grid_search.best_estimator_)
print()
print("Best CV Score:",grid_search.best_score_)

if APPLY_SCALER:
    X_test_scaled = scaler.transform(X_test)
else:
    X_test_scaled = X_test

# Prediction
y_pred = best_knn.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

if SHOW_CONFMATRIX:
    ConfusionMatrixDisplay.from_estimator(best_knn, X_test_scaled, y_test)

Fitting 5 folds for each of 28 candidates, totalling 140 fits

Best Param: KNeighborsClassifier(n_neighbors=3, weights='distance')

Best CV Score: 0.47310118088448194
              precision    recall  f1-score   support

         0.0       0.87      0.93      0.90      1607
         1.0       0.60      0.43      0.50       393

    accuracy                           0.83      2000
   macro avg       0.73      0.68      0.70      2000
weighted avg       0.82      0.83      0.82      2000



##Decision Tree

In [48]:
dt_clf = DecisionTreeClassifier(random_state=42)

# Training
dt_clf.fit(X_res, y_res)

if APPLY_SCALER:
    X_test_scaled = scaler.transform(X_test)
else:
    X_test_scaled = X_test

# Prediction
y_pred = dt_clf.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

if SHOW_CONFMATRIX:
    ConfusionMatrixDisplay.from_estimator(dt_clf, X_test_scaled, y_test)

              precision    recall  f1-score   support

         0.0       0.87      0.85      0.86      1607
         1.0       0.44      0.50      0.47       393

    accuracy                           0.78      2000
   macro avg       0.66      0.67      0.66      2000
weighted avg       0.79      0.78      0.78      2000



In [49]:
param_grid = {
    "n_neighbors": [1,3,5,7,11,15,100],
    "weights": ["uniform", "distance"],
    # "metric": ["euclidean", "manhattan"]
    "p":[1,2]
}

dt_clf = DecisionTreeClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
    )

grid_search.fit(X_res, y_res)

best_knn = grid_search.best_estimator_

print()
print("Best Param:",grid_search.best_estimator_)
print()
print("Best CV Score:",grid_search.best_score_)

if APPLY_SCALER:
    X_test_scaled = scaler.transform(X_test)
else:
    X_test_scaled = X_test

# Prediction
y_pred = best_knn.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

if SHOW_CONFMATRIX:
    ConfusionMatrixDisplay.from_estimator(best_knn, X_test_scaled, y_test)

Fitting 5 folds for each of 28 candidates, totalling 140 fits

Best Param: KNeighborsClassifier(n_neighbors=3, weights='distance')

Best CV Score: 0.47310118088448194
              precision    recall  f1-score   support

         0.0       0.87      0.93      0.90      1607
         1.0       0.60      0.43      0.50       393

    accuracy                           0.83      2000
   macro avg       0.73      0.68      0.70      2000
weighted avg       0.82      0.83      0.82      2000

